# Lớp 3 — Analytics & Master Table

Notebook này chạy **sau** Lớp 1 và Lớp 2. Nó không chạm vào raw data mà chỉ đọc
các bảng đã được tạo sẵn, rồi tổng hợp thành một bộ bảng cuối cùng phục vụ phân tích.

Đầu vào:
- `analytics/sessions.csv` — session metadata, phân loại từ Lớp 1
- `analytics/protocol_phases.csv` — timestamps pre-rest/meditation/post-rest
- `analytics/layer2/session_label.csv` — nhãn signal QC từ Lớp 2
- `analytics/layer2/signal_qc_full.csv` — metrics chi tiết từng kênh

Đầu ra (`analytics/layer3/`):
- `master_sessions.csv/parquet` — 1 hàng/session, gộp mọi thứ
- `participant_summary.csv` — yield, coverage, modality failure rate per participant
- `study_day_summary.csv` — tổng hợp theo ngày nghiên cứu
- `analysis_roster.csv` — danh sách session đủ điều kiện đưa vào phân tích chính
- `flags_for_review.csv` — danh sách các trường hợp cần xem lại
- `study_overview.png` — biểu đồ tổng quan study

## 1. Cấu hình

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

PROJECT_DIR  = Path("../..") / "data"
ANALYTICS_DIR = PROJECT_DIR / "analytics"
LAYER2_DIR    = ANALYTICS_DIR / "layer2"
LAYER3_DIR    = ANALYTICS_DIR / "layer3"
LAYER3_DIR.mkdir(parents=True, exist_ok=True)

# Ngưỡng để đưa session vào analysis_roster
# Thay đổi tuỳ protocol nghiên cứu
ROSTER_REQUIRE_PASS       = False   # True = chỉ nhận PASS; False = nhận cả CONDITIONAL_PASS
MIN_STUDY_DAYS_PER_PART   = 1       # Participant cần ít nhất N ngày hợp lệ

print("Layer 3 output:", LAYER3_DIR)

## 2. Đọc dữ liệu từ Lớp 1 và Lớp 2

In [ ]:
def read_if_exists(path: Path, label: str) -> pd.DataFrame:
    if path.exists():
        try:
            df = pd.read_csv(path, low_memory=False)
        except pd.errors.EmptyDataError:
            print(f"  [EMPTY] {label}: {path.name}")
            return pd.DataFrame()
        print(f"  {label}: {len(df)} rows — {path.name}")
        return df
    print(f"  [MISSING] {label}: {path}")
    return pd.DataFrame()

print("Đọc Lớp 1:")
l1_sessions = read_if_exists(ANALYTICS_DIR / "layer1" / "sessions.csv",         "sessions")
l1_phases   = read_if_exists(ANALYTICS_DIR / "layer1" / "protocol_phases.csv",  "protocol_phases")
l1_day_sum  = read_if_exists(ANALYTICS_DIR / "layer1" / "day_summary.csv",      "day_summary")
l1_sig_qc   = read_if_exists(ANALYTICS_DIR / "layer1" / "signal_qc_session.csv","signal_qc_session")

print("\nĐọc Lớp 2:")
l2_labels   = read_if_exists(LAYER2_DIR / "session_label.csv",   "session_label")
l2_sig_qc   = read_if_exists(LAYER2_DIR / "signal_qc_full.csv",  "signal_qc_full")
l2_run_log  = read_if_exists(LAYER2_DIR / "layer2_run_log.csv",  "run_log")

## 3. Tạo master_sessions — 1 hàng/session

In [ ]:
# Pivot signal_qc để có 1 hàng/session thay vì 5 hàng/session
def pivot_signal_qc(sig_qc: pd.DataFrame) -> pd.DataFrame:
    """Chuyển từ long format (5 rows/session) sang wide (1 row/session)."""
    if sig_qc.empty or "signal" not in sig_qc.columns:
        return pd.DataFrame()

    metric_cols = ["label", "artifact_window_percent", "missing_percent",
                   "longest_flatline_seconds", "filtered_rms", "duration_seconds"]
    metric_cols = [c for c in metric_cols if c in sig_qc.columns]

    rows = []
    for sid, grp in sig_qc.groupby("session_id"):
        row = {"session_id": sid}
        for _, sig_row in grp.iterrows():
            sig = sig_row["signal"].replace(" ", "_").lower()
            for col in metric_cols:
                row[f"sig_{sig}_{col}"] = sig_row.get(col)
        rows.append(row)
    return pd.DataFrame(rows)


sig_wide = pivot_signal_qc(l2_sig_qc)

# Chọn cột Lớp 2 labels cần gộp (loại cột trùng với Lớp 1)
L2_LABEL_COLS = [
    "session_id",
    "recording_label", "eeg_label", "ppg_label", "fnirs_raw_label",
    "fnirs_hbo_hbr_readiness", "overall_session_label",
    "global_multimodal_artifact_percent", "analysis_status",
    "longest_block_seconds", "n_blocks", "eeg_rms_ratio",
    "fnirs_raw_correlation", "reasons",
]
l2_labels_slim = l2_labels[[c for c in L2_LABEL_COLS if c in l2_labels.columns]] if not l2_labels.empty else pd.DataFrame()

# Gộp Lớp 1 + Lớp 2
master = l1_sessions.copy()

if not l2_labels_slim.empty:
    master = master.merge(l2_labels_slim, on="session_id", how="left", suffixes=("_l1", "_l2"))

if not sig_wide.empty:
    master = master.merge(sig_wide, on="session_id", how="left")

# Tạo cột quyết định cuối cùng kết hợp cả hai lớp
# Lớp 1 FAIL → session không chạy Lớp 2 → final_label = EXCLUDED_L1
# Lớp 1 PASS nhưng Lớp 2 FAIL → EXCLUDED_L2_DURATION hoặc EXCLUDED_L2_SIGNAL
# Cả hai PASS → INCLUDED hoặc INCLUDED_CONDITIONAL
#
# EXCLUDED_L2 được tách làm 2 vì 2 nguyên nhân khác hẳn nhau về bản chất:
# recording ngắn hơn ngưỡng tối thiểu (đầy đủ giao thức nhưng chưa hoàn thành)
# so với tín hiệu thực sự nhiễu/hỏng (đủ thời lượng nhưng chất lượng kém).
# Gộp chung khiến reviewer mất công tra tín hiệu cho những session vốn chỉ
# bị loại vì ngắn -- điều mà eeg_label/ppg_label/fnirs_raw_label bên dưới đã
# thường là PASS/CONDITIONAL_PASS.
def final_label(row):
    if not row.get("analysis_eligible", False):
        return "EXCLUDED_L1"
    l2 = row.get("overall_session_label")
    if pd.isna(l2):
        return "L2_NOT_RUN"
    if l2 == "FAIL":
        reasons = str(row.get("reasons", ""))
        if "longest_continuous_block_below_minimum" in reasons:
            return "EXCLUDED_L2_DURATION"
        return "EXCLUDED_L2_SIGNAL"
    if l2 == "CONDITIONAL_PASS":
        return "INCLUDED_CONDITIONAL"
    if l2 == "PASS":
        return "INCLUDED"
    return "UNKNOWN"

master["final_label"] = master.apply(final_label, axis=1)

# Sắp xếp cột hợp lý
priority_cols = [
    "participant_key", "participant_name_from_zip", "session_date_from_zip",
    "study_day", "session_id", "session_role", "analysis_eligible",
    "final_label", "overall_session_label",
    "eeg_label", "ppg_label", "fnirs_raw_label",
    "global_multimodal_artifact_percent", "reasons",
    "start_time", "result_duration_seconds",
    "protocol_wall_seconds", "csv_duration_packet_seconds",
]
other_cols = [c for c in master.columns if c not in priority_cols]
priority_cols_present = [c for c in priority_cols if c in master.columns]
master = master[priority_cols_present + other_cols]

master.to_csv(LAYER3_DIR / "master_sessions.csv", index=False)
try:
    master.to_parquet(LAYER3_DIR / "master_sessions.parquet", index=False)
except Exception as e:
    print("Parquet skip:", e)

print(f"Master table: {len(master)} sessions × {len(master.columns)} columns")
print("\nPhân bố final_label:")
print(master["final_label"].value_counts().to_string())

## 4. Participant summary

In [ ]:
def participant_summary(master: pd.DataFrame) -> pd.DataFrame:
    """1 hàng / participant. Tính yield, coverage, modality failure rates."""
    if master.empty:
        return pd.DataFrame()

    rows = []
    for pkey, grp in master.groupby("participant_key", dropna=False):
        primary = grp[grp["session_role"] == "primary_valid"]
        included = grp[grp["final_label"].isin(["INCLUDED", "INCLUDED_CONDITIONAL"])]
        included_clean = grp[grp["final_label"] == "INCLUDED"]

        n_days_attempted = primary["study_day"].nunique()
        n_days_included  = included["study_day"].nunique()
        n_days_pass_only = included_clean["study_day"].nunique()

        def label_rate(col, label):
            if col not in primary.columns or primary.empty:
                return float("nan")
            return float((primary[col] == label).mean() * 100)

        row = {
            "participant_key": pkey,
            "participant_name": grp["participant_name_from_zip"].iloc[0]
                if "participant_name_from_zip" in grp.columns else None,
            "group_name": grp["group_name"].mode().iloc[0]
                if "group_name" in grp.columns and not grp["group_name"].isna().all() else None,
            "total_sessions": len(grp),
            "primary_valid_sessions": len(primary),
            "invalid_attempt_sessions": int((grp["session_role"] == "invalid_attempt").sum()),
            "days_attempted": n_days_attempted,
            "days_included": n_days_included,
            "days_pass_only": n_days_pass_only,
            "yield_percent": round(n_days_included / n_days_attempted * 100, 1)
                if n_days_attempted > 0 else float("nan"),
            "eeg_fail_rate_percent": label_rate("eeg_label", "FAIL"),
            "ppg_fail_rate_percent": label_rate("ppg_label", "FAIL"),
            "fnirs_fail_rate_percent": label_rate("fnirs_raw_label", "FAIL"),
            "any_excluded_l2": int(grp["final_label"].isin(["EXCLUDED_L2_SIGNAL", "EXCLUDED_L2_DURATION"]).sum()),
            "study_days_covered": sorted(grp["study_day"].dropna().astype(int).unique().tolist()),
            "missing_days": [],  # isi di bawah
        }

        # Ngày nào trong dải có dữ liệu nhưng không có session hợp lệ
        if n_days_attempted > 0:
            covered = set(grp["study_day"].dropna().astype(int))
            full_range = set(range(int(grp["study_day"].min()), int(grp["study_day"].max()) + 1))
            row["missing_days"] = sorted(full_range - covered)

        rows.append(row)

    return pd.DataFrame(rows).sort_values("participant_key")


part_sum = participant_summary(master)
part_sum.to_csv(LAYER3_DIR / "participant_summary.csv", index=False)

display(part_sum[[
    "participant_name", "group_name",
    "days_attempted", "days_included", "yield_percent",
    "eeg_fail_rate_percent", "ppg_fail_rate_percent", "fnirs_fail_rate_percent",
    "any_excluded_l2", "missing_days"
]])

## 5. Study-day summary

In [ ]:
def study_day_summary(master: pd.DataFrame) -> pd.DataFrame:
    """1 hàng / study_day. Số participant, yield, modality rates."""
    if master.empty:
        return pd.DataFrame()

    rows = []
    for day, grp in master.groupby("study_day", dropna=False):
        primary = grp[grp["session_role"] == "primary_valid"]
        included = grp[grp["final_label"].isin(["INCLUDED", "INCLUDED_CONDITIONAL"])]

        def label_rate(col, label):
            if col not in primary.columns or primary.empty:
                return float("nan")
            return round(float((primary[col] == label).mean() * 100), 1)

        rows.append({
            "study_day": day,
            "n_participants": grp["participant_key"].nunique(),
            "n_primary_valid": len(primary),
            "n_included": len(included),
            "n_excluded_l2": int(grp["final_label"].isin(["EXCLUDED_L2_SIGNAL", "EXCLUDED_L2_DURATION"]).sum()),
            "yield_percent": round(len(included) / len(primary) * 100, 1)
                if len(primary) > 0 else float("nan"),
            "eeg_pass_rate": label_rate("eeg_label", "PASS"),
            "ppg_pass_rate": label_rate("ppg_label", "PASS"),
            "fnirs_pass_rate": label_rate("fnirs_raw_label", "PASS"),
            "mean_multimodal_artifact_pct":
                round(float(primary["global_multimodal_artifact_percent"].mean()), 1)
                if "global_multimodal_artifact_percent" in primary.columns else float("nan"),
        })

    return pd.DataFrame(rows).sort_values("study_day")


day_sum = study_day_summary(master)
day_sum.to_csv(LAYER3_DIR / "study_day_summary.csv", index=False)
display(day_sum)

## 6. Analysis roster — danh sách session đưa vào phân tích chính

In [ ]:
# Điều kiện đưa vào roster
if ROSTER_REQUIRE_PASS:
    roster_mask = master["final_label"] == "INCLUDED"
else:
    roster_mask = master["final_label"].isin(["INCLUDED", "INCLUDED_CONDITIONAL"])

roster = master[roster_mask].copy()

# Kiểm tra participant đủ số ngày tối thiểu
days_per_part = roster.groupby("participant_key")["study_day"].nunique()
enough_days = days_per_part[days_per_part >= MIN_STUDY_DAYS_PER_PART].index
roster = roster[roster["participant_key"].isin(enough_days)]

roster_cols = [
    "participant_key", "participant_name_from_zip", "group_name",
    "study_day", "session_date_from_zip", "session_id",
    "final_label", "overall_session_label",
    "eeg_label", "ppg_label", "fnirs_raw_label",
    "longest_block_seconds", "global_multimodal_artifact_percent",
    "start_time", "result_duration_seconds",
]
roster_out = roster[[c for c in roster_cols if c in roster.columns]].copy()
roster_out = roster_out.sort_values(["participant_key", "study_day"])
roster_out.to_csv(LAYER3_DIR / "analysis_roster.csv", index=False)

print(f"Analysis roster: {len(roster_out)} sessions")
print(f"Participants: {roster_out['participant_key'].nunique()}")
print("\nPhân bố final_label:")
print(roster_out["final_label"].value_counts().to_string())
display(roster_out)

## 7. Flags cần review

In [ ]:
flags_rows = []

# Flag 1a: Session bị loại vì tín hiệu nhiễu/hỏng -- cần review kỹ.
for _, row in master[master["final_label"] == "EXCLUDED_L2_SIGNAL"].iterrows():
    flags_rows.append({
        "flag_type": "EXCLUDED_L2_SIGNAL",
        "priority": "HIGH",
        "participant_key": row["participant_key"],
        "study_day": row.get("study_day"),
        "session_id": row["session_id"],
        "detail": row.get("reasons", ""),
    })

# Flag 1b: Session bị loại chỉ vì recording ngắn hơn ngưỡng tối thiểu --
# thường tín hiệu vẫn ổn (xem eeg_label/ppg_label/fnirs_raw_label), không
# cần review chất lượng tín hiệu, chỉ cần theo dõi việc hoàn thành protocol.
for _, row in master[master["final_label"] == "EXCLUDED_L2_DURATION"].iterrows():
    flags_rows.append({
        "flag_type": "EXCLUDED_L2_DURATION",
        "priority": "LOW",
        "participant_key": row["participant_key"],
        "study_day": row.get("study_day"),
        "session_id": row["session_id"],
        "detail": row.get("reasons", ""),
    })

# Flag 2: Participant có >= 2 ngày bị loại liên tiếp (bất kỳ lý do nào)
for pkey, grp in master.groupby("participant_key"):
    day_labels = grp.sort_values("study_day")[["study_day", "final_label"]].dropna()
    excl = day_labels["final_label"].isin(
        ["EXCLUDED_L1", "EXCLUDED_L2_DURATION", "EXCLUDED_L2_SIGNAL"]
    ).tolist()
    for i in range(len(excl) - 1):
        if excl[i] and excl[i + 1]:
            flags_rows.append({
                "flag_type": "CONSECUTIVE_EXCLUSIONS",
                "priority": "HIGH",
                "participant_key": pkey,
                "study_day": int(day_labels.iloc[i]["study_day"]),
                "session_id": None,
                "detail": f"Day {int(day_labels.iloc[i]['study_day'])} and {int(day_labels.iloc[i+1]['study_day'])} both excluded",
            })
            break

# Flag 3: Participant thiếu ngày (gap trong study period)
for _, prow in part_sum.iterrows():
    if prow["missing_days"]:
        flags_rows.append({
            "flag_type": "MISSING_DAYS",
            "priority": "MEDIUM",
            "participant_key": prow["participant_key"],
            "study_day": None,
            "session_id": None,
            "detail": f"Missing days: {prow['missing_days']}",
        })

# Flag 4: fNIRS correlation thấp
if "fnirs_raw_correlation" in master.columns:
    for _, row in master[
        master["fnirs_raw_correlation"].notna() &
        (master["fnirs_raw_correlation"] < 0.5)
    ].iterrows():
        flags_rows.append({
            "flag_type": "LOW_FNIRS_CORRELATION",
            "priority": "MEDIUM",
            "participant_key": row["participant_key"],
            "study_day": row.get("study_day"),
            "session_id": row["session_id"],
            "detail": f"fNIRS corr = {row['fnirs_raw_correlation']:.3f}",
        })

# Flag 5: EEG channel imbalance nghiêm trọng
if "eeg_rms_ratio" in master.columns:
    for _, row in master[
        master["eeg_rms_ratio"].notna() & (master["eeg_rms_ratio"] > 10)
    ].iterrows():
        flags_rows.append({
            "flag_type": "EEG_CHANNEL_IMBALANCE",
            "priority": "MEDIUM",
            "participant_key": row["participant_key"],
            "study_day": row.get("study_day"),
            "session_id": row["session_id"],
            "detail": f"EEG RMS ratio = {row['eeg_rms_ratio']:.1f}",
        })

flags_df = pd.DataFrame(flags_rows).drop_duplicates()
if flags_df.empty:
    flags_df = pd.DataFrame(columns=["flag_type", "priority", "participant_key", "study_day", "session_id", "detail"])
flags_df = flags_df.sort_values(["priority", "flag_type", "participant_key"])
flags_df.to_csv(LAYER3_DIR / "flags_for_review.csv", index=False)

print(f"Flags: {len(flags_df)} tổng cộng")
if not flags_df.empty:
    print(flags_df["flag_type"].value_counts().to_string())
    display(flags_df)

## 8. Biểu đồ tổng quan study

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

COLOR_MAP = {
    "INCLUDED": "#2ecc71",
    "INCLUDED_CONDITIONAL": "#f39c12",
    "EXCLUDED_L2_SIGNAL": "#e74c3c",
    "EXCLUDED_L2_DURATION": "#5b8def",
    "EXCLUDED_L1": "#bdc3c7",
    "L2_NOT_RUN": "#95a5a6",
    "UNKNOWN": "#ecf0f1",
}

LABEL_SHORT_MAP = {
    "INCLUDED": "IN",
    "INCLUDED_CONDITIONAL": "COND",
    "EXCLUDED_L2_SIGNAL": "SIG",
    "EXCLUDED_L2_DURATION": "DUR",
    "EXCLUDED_L1": "L1",
    "L2_NOT_RUN": "N/R",
    "UNKNOWN": "?",
}

YIELD_TARGET_PERCENT = 80

# LAYER3_DIR phải được khai báo từ cell trước.
# Có thể dùng đoạn dưới nếu chưa có:
#
# LAYER3_DIR = Path("../..") / "data" / "analytics" / "layer3"

LAYER3_DIR = Path(LAYER3_DIR)
LAYER3_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = LAYER3_DIR / "study_overview.png"


# ============================================================
# VALIDATION
# ============================================================

required_master_columns = {
    "participant_key",
    "study_day",
    "session_role",
}

missing_master_columns = required_master_columns.difference(master.columns)

if missing_master_columns:
    raise KeyError(
        "master đang thiếu các cột bắt buộc: "
        + ", ".join(sorted(missing_master_columns))
    )


# ============================================================
# PREPARE MASTER DATA
# ============================================================

master_plot = master.copy()

# Participant key luôn được chuẩn hóa thành string.
master_plot["participant_key_clean"] = (
    master_plot["participant_key"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
)

master_plot.loc[
    master_plot["participant_key_clean"].isin(
        ["", "nan", "None", "<NA>"]
    ),
    "participant_key_clean",
] = "UNKNOWN"

# Study day chuyển về numeric nullable integer.
master_plot["study_day_num"] = pd.to_numeric(
    master_plot["study_day"],
    errors="coerce",
).astype("Int64")

# Session role chuẩn hóa thành string.
master_plot["session_role_clean"] = (
    master_plot["session_role"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
)

# Final label chuẩn hóa.
if "final_label" not in master_plot.columns:
    master_plot["final_label_clean"] = "UNKNOWN"
else:
    master_plot["final_label_clean"] = (
        master_plot["final_label"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )

master_plot.loc[
    ~master_plot["final_label_clean"].isin(COLOR_MAP.keys()),
    "final_label_clean",
] = "UNKNOWN"

# Chỉ giữ primary valid và day hợp lệ.
primary_only = master_plot[
    master_plot["session_role_clean"].eq("primary_valid")
    & master_plot["study_day_num"].notna()
].copy()


# ============================================================
# PREPARE DAY SUMMARY
# ============================================================

if day_sum is None:
    day_sum_plot = pd.DataFrame()
else:
    day_sum_plot = day_sum.copy()

if not day_sum_plot.empty:
    if "study_day" in day_sum_plot.columns:
        day_sum_plot["study_day_num"] = pd.to_numeric(
            day_sum_plot["study_day"],
            errors="coerce",
        ).astype("Int64")

        day_sum_plot = (
            day_sum_plot[
                day_sum_plot["study_day_num"].notna()
            ]
            .sort_values("study_day_num")
            .reset_index(drop=True)
        )

    if "yield_percent" in day_sum_plot.columns:
        day_sum_plot["yield_percent_num"] = pd.to_numeric(
            day_sum_plot["yield_percent"],
            errors="coerce",
        ).fillna(0)

        day_sum_plot["yield_percent_num"] = (
            day_sum_plot["yield_percent_num"]
            .clip(lower=0, upper=100)
        )


# ============================================================
# PREPARE PARTICIPANT SUMMARY
# ============================================================

if part_sum is None:
    part_sum_plot = pd.DataFrame()
else:
    part_sum_plot = part_sum.copy()

if not part_sum_plot.empty:
    if "participant_key" in part_sum_plot.columns:
        part_sum_plot["participant_key_clean"] = (
            part_sum_plot["participant_key"]
            .fillna("UNKNOWN")
            .astype(str)
            .str.strip()
        )

        part_sum_plot.loc[
            part_sum_plot["participant_key_clean"].isin(
                ["", "nan", "None", "<NA>"]
            ),
            "participant_key_clean",
        ] = "UNKNOWN"

    if "yield_percent" in part_sum_plot.columns:
        part_sum_plot["yield_percent_num"] = pd.to_numeric(
            part_sum_plot["yield_percent"],
            errors="coerce",
        ).fillna(0)

        part_sum_plot["yield_percent_num"] = (
            part_sum_plot["yield_percent_num"]
            .clip(lower=0, upper=100)
        )

    fail_rate_columns = [
        "eeg_fail_rate_percent",
        "ppg_fail_rate_percent",
        "fnirs_fail_rate_percent",
    ]

    for column in fail_rate_columns:
        if column in part_sum_plot.columns:
            part_sum_plot[column] = pd.to_numeric(
                part_sum_plot[column],
                errors="coerce",
            )

    if "participant_key_clean" in part_sum_plot.columns:
        part_sum_plot = (
            part_sum_plot
            .sort_values("participant_key_clean")
            .reset_index(drop=True)
        )


# ============================================================
# DRAW FIGURE
# ============================================================

if primary_only.empty:
    print("Không có primary_valid session để vẽ.")

else:
    participants = sorted(
        primary_only["participant_key_clean"]
        .dropna()
        .unique()
        .tolist()
    )

    days = sorted(
        primary_only["study_day_num"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if not participants:
        print("Không có participant hợp lệ để vẽ.")

    elif not days:
        print("Không có study day hợp lệ để vẽ.")

    else:
        figure_width = max(
            10,
            len(days) * 1.05 + 4,
        )

        figure_height = max(
            6,
            len(participants) * 0.72 + 4,
        )

        fig = plt.figure(
            figsize=(figure_width, figure_height)
        )

        gs = gridspec.GridSpec(
            nrows=2,
            ncols=2,
            height_ratios=[3, 1.25],
            width_ratios=[3.2, 1.2],
            hspace=0.5,
            wspace=0.38,
        )

        # ====================================================
        # 1. HEATMAP
        # ====================================================

        ax_heat = fig.add_subplot(gs[0, 0])

        for participant_index, participant_key in enumerate(
            participants
        ):
            for day_index, study_day in enumerate(days):
                cell = primary_only[
                    primary_only[
                        "participant_key_clean"
                    ].eq(participant_key)
                    & primary_only[
                        "study_day_num"
                    ].eq(study_day)
                ]

                if cell.empty:
                    label = None
                    color = "#f5f5f5"
                    display_text = ""

                else:
                    # Nếu có nhiều primary_valid trong cùng participant-day,
                    # lấy dòng đầu tiên theo thứ tự dữ liệu.
                    label = (
                        cell.iloc[0]
                        .get(
                            "final_label_clean",
                            "UNKNOWN",
                        )
                    )

                    if label not in COLOR_MAP:
                        label = "UNKNOWN"

                    color = COLOR_MAP[label]
                    display_text = LABEL_SHORT_MAP.get(
                        label,
                        "?",
                    )

                rectangle = plt.Rectangle(
                    (
                        day_index - 0.45,
                        participant_index - 0.45,
                    ),
                    0.9,
                    0.9,
                    facecolor=color,
                    edgecolor="white",
                    linewidth=1.5,
                )

                ax_heat.add_patch(rectangle)

                if display_text:
                    text_color = (
                        "white"
                        if label in {
                            "INCLUDED",
                            "EXCLUDED_L2_SIGNAL",
                        }
                        else "#2c3e50"
                    )

                    ax_heat.text(
                        day_index,
                        participant_index,
                        display_text,
                        ha="center",
                        va="center",
                        fontsize=6.5,
                        color=text_color,
                        fontweight="bold",
                    )

        ax_heat.set_xlim(
            -0.5,
            len(days) - 0.5,
        )

        ax_heat.set_ylim(
            -0.5,
            len(participants) - 0.5,
        )

        ax_heat.set_xticks(
            range(len(days))
        )

        ax_heat.set_xticklabels(
            [f"Day {day}" for day in days],
            fontsize=8,
            rotation=0,
        )

        participant_heat_labels = [
            (
                str(participant)[:12]
                if pd.notna(participant)
                else "UNKNOWN"
            )
            for participant in participants
        ]

        ax_heat.set_yticks(
            range(len(participants))
        )

        ax_heat.set_yticklabels(
            participant_heat_labels,
            fontsize=7,
        )

        ax_heat.set_title(
            "Session QC outcome per participant × day",
            fontsize=11,
            pad=10,
        )

        ax_heat.set_xlabel(
            "Study day",
            fontsize=9,
        )

        ax_heat.set_ylabel(
            "Participant",
            fontsize=9,
        )

        ax_heat.set_aspect("equal")

        ax_heat.tick_params(
            axis="both",
            length=0,
        )

        for spine in ax_heat.spines.values():
            spine.set_visible(False)

        legend_elements = [
            Patch(
                facecolor=color,
                edgecolor="white",
                label=label,
            )
            for label, color in COLOR_MAP.items()
        ]

        ax_heat.legend(
            handles=legend_elements,
            fontsize=7,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.13),
            ncol=3,
            frameon=False,
        )

        # ====================================================
        # 2. YIELD PER DAY
        # ====================================================

        ax_day_bar = fig.add_subplot(gs[1, 0])

        day_chart_available = (
            not day_sum_plot.empty
            and "study_day_num" in day_sum_plot.columns
            and "yield_percent_num" in day_sum_plot.columns
        )

        if day_chart_available:
            x_positions = np.arange(
                len(day_sum_plot)
            )

            day_yield_values = (
                day_sum_plot["yield_percent_num"]
                .to_numpy(dtype=float)
            )

            day_bars = ax_day_bar.bar(
                x_positions,
                day_yield_values,
                color="#3498db",
                alpha=0.75,
                edgecolor="white",
                linewidth=0.6,
            )

            ax_day_bar.set_xticks(
                x_positions
            )

            ax_day_bar.set_xticklabels(
                [
                    f"D{int(day)}"
                    for day in day_sum_plot[
                        "study_day_num"
                    ]
                ],
                fontsize=8,
            )

            ax_day_bar.set_ylabel(
                "Yield (%)",
                fontsize=8,
            )

            ax_day_bar.set_ylim(
                0,
                105,
            )

            ax_day_bar.axhline(
                YIELD_TARGET_PERCENT,
                color="#f39c12",
                linestyle="--",
                linewidth=1,
                label=(
                    f"Target "
                    f"{YIELD_TARGET_PERCENT}%"
                ),
            )

            ax_day_bar.set_title(
                "Yield per study day",
                fontsize=9,
            )

            ax_day_bar.grid(
                axis="y",
                alpha=0.25,
                linestyle=":",
            )

            ax_day_bar.legend(
                fontsize=7,
                frameon=False,
                loc="upper right",
            )

            for bar, value in zip(
                day_bars,
                day_yield_values,
            ):
                ax_day_bar.text(
                    bar.get_x()
                    + bar.get_width() / 2,
                    min(value + 2, 102),
                    f"{value:.0f}",
                    ha="center",
                    va="bottom",
                    fontsize=6.5,
                )

        else:
            ax_day_bar.text(
                0.5,
                0.5,
                "No day-level yield data",
                ha="center",
                va="center",
                transform=ax_day_bar.transAxes,
                fontsize=9,
                color="#7f8c8d",
            )

            ax_day_bar.set_axis_off()

        # ====================================================
        # 3. YIELD PER PARTICIPANT
        # ====================================================

        ax_participant_bar = fig.add_subplot(
            gs[0, 1]
        )

        participant_chart_available = (
            not part_sum_plot.empty
            and "participant_key_clean"
            in part_sum_plot.columns
            and "yield_percent_num"
            in part_sum_plot.columns
        )

        if participant_chart_available:
            participant_positions = np.arange(
                len(part_sum_plot)
            )

            participant_yield_values = (
                part_sum_plot["yield_percent_num"]
                .to_numpy(dtype=float)
            )

            participant_bars = (
                ax_participant_bar.barh(
                    participant_positions,
                    participant_yield_values,
                    color="#9b59b6",
                    alpha=0.75,
                    edgecolor="white",
                    linewidth=0.6,
                )
            )

            participant_labels = (
                part_sum_plot[
                    "participant_key_clean"
                ]
                .fillna("UNKNOWN")
                .astype(str)
                .str[:10]
                .tolist()
            )

            ax_participant_bar.set_yticks(
                participant_positions
            )

            ax_participant_bar.set_yticklabels(
                participant_labels,
                fontsize=7,
            )

            ax_participant_bar.set_xlabel(
                "Yield (%)",
                fontsize=8,
            )

            ax_participant_bar.set_xlim(
                0,
                105,
            )

            ax_participant_bar.axvline(
                YIELD_TARGET_PERCENT,
                color="#f39c12",
                linestyle="--",
                linewidth=1,
            )

            ax_participant_bar.set_title(
                "Yield per participant",
                fontsize=9,
            )

            ax_participant_bar.grid(
                axis="x",
                alpha=0.25,
                linestyle=":",
            )

            ax_participant_bar.invert_yaxis()

            for bar, value in zip(
                participant_bars,
                participant_yield_values,
            ):
                ax_participant_bar.text(
                    min(value + 1.5, 101),
                    bar.get_y()
                    + bar.get_height() / 2,
                    f"{value:.0f}",
                    va="center",
                    ha="left",
                    fontsize=6.5,
                )

        else:
            ax_participant_bar.text(
                0.5,
                0.5,
                "No participant-level yield data",
                ha="center",
                va="center",
                transform=ax_participant_bar.transAxes,
                fontsize=9,
                color="#7f8c8d",
            )

            ax_participant_bar.set_axis_off()

        # ====================================================
        # 4. MODALITY FAIL RATES
        # ====================================================

        ax_modality = fig.add_subplot(
            gs[1, 1]
        )

        modality_config = {
            "eeg_fail_rate_percent": {
                "label": "EEG",
                "color": "#e74c3c",
            },
            "ppg_fail_rate_percent": {
                "label": "PPG",
                "color": "#e67e22",
            },
            "fnirs_fail_rate_percent": {
                "label": "fNIRS",
                "color": "#8e44ad",
            },
        }

        available_modality_columns = [
            column
            for column in modality_config
            if (
                column in part_sum_plot.columns
                and part_sum_plot[column]
                .notna()
                .any()
            )
        ]

        if (
            not part_sum_plot.empty
            and available_modality_columns
        ):
            modality_labels = []
            modality_means = []
            modality_colors = []

            for column in available_modality_columns:
                values = pd.to_numeric(
                    part_sum_plot[column],
                    errors="coerce",
                )

                mean_value = values.mean(
                    skipna=True
                )

                if pd.isna(mean_value):
                    continue

                modality_labels.append(
                    modality_config[column]["label"]
                )

                modality_means.append(
                    float(
                        np.clip(
                            mean_value,
                            0,
                            100,
                        )
                    )
                )

                modality_colors.append(
                    modality_config[column]["color"]
                )

            if modality_labels:
                modality_bars = ax_modality.bar(
                    modality_labels,
                    modality_means,
                    color=modality_colors,
                    alpha=0.78,
                    edgecolor="white",
                    linewidth=0.6,
                )

                ax_modality.set_ylabel(
                    "Fail rate (%)",
                    fontsize=8,
                )

                ax_modality.set_title(
                    "Mean modality fail rate",
                    fontsize=9,
                )

                ax_modality.set_ylim(
                    0,
                    105,
                )

                ax_modality.grid(
                    axis="y",
                    alpha=0.25,
                    linestyle=":",
                )

                for bar, value in zip(
                    modality_bars,
                    modality_means,
                ):
                    ax_modality.text(
                        bar.get_x()
                        + bar.get_width() / 2,
                        min(value + 2, 102),
                        f"{value:.1f}",
                        ha="center",
                        va="bottom",
                        fontsize=6.5,
                    )

            else:
                ax_modality.text(
                    0.5,
                    0.5,
                    "No modality fail-rate data",
                    ha="center",
                    va="center",
                    transform=ax_modality.transAxes,
                    fontsize=9,
                    color="#7f8c8d",
                )

                ax_modality.set_axis_off()

        else:
            ax_modality.text(
                0.5,
                0.5,
                "No modality fail-rate data",
                ha="center",
                va="center",
                transform=ax_modality.transAxes,
                fontsize=9,
                color="#7f8c8d",
            )

            ax_modality.set_axis_off()

        # ====================================================
        # FINAL FORMATTING
        # ====================================================

        fig.suptitle(
            "Meditation study — QC overview",
            fontsize=13,
            fontweight="bold",
            y=0.995,
        )

        plt.tight_layout(
            rect=[0, 0, 1, 0.97]
        )

        fig.savefig(
            OUTPUT_PATH,
            dpi=180,
            bbox_inches="tight",
            facecolor="white",
        )

        plt.show()

        print(f"Saved: {OUTPUT_PATH}")

## 9. Tổng kết

In [ ]:
print("=" * 55)
print("LAYER 3 ANALYTICS — HOÀN THÀNH")
print("=" * 55)

total = len(master)
n_included = int(master["final_label"].isin(["INCLUDED", "INCLUDED_CONDITIONAL"]).sum())
n_incl_clean = int((master["final_label"] == "INCLUDED").sum())
n_roster = len(roster_out)
n_flags = len(flags_df)
n_parts = master["participant_key"].nunique()

print(f"Participants:       {n_parts}")
print(f"Total sessions:     {total}")
print(f"Included (any):     {n_included}  ({100*n_included/total:.0f}%" if total else "")
print(f"Included (PASS):    {n_incl_clean}")
print(f"Analysis roster:    {n_roster} sessions")
print(f"Flags for review:   {n_flags}")

print(f"\nOutput: {LAYER3_DIR}")
for f in sorted(LAYER3_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")